# Limpieza del dataset de notas

**Objetivo:** limpiar el registro de actividades evaluadas exportado de Canvas (una fila por actividad por estudiante) para que, al sumar los puntos de cada estudiante, se obtenga su nota final real y se puedan construir características mensuales de desempeño.

| | |
|---|---|
| **Entradas** | `Datos/pensamiento.RData` (objeto `pensamiento_final`, 61,943 filas), `Datos/registro_cleaned.csv` (salida de [`Limpieza_registro.ipynb`](Limpieza_registro.ipynb)) y `Datos/Actividades Pensamiento.xlsx` (planificación oficial de actividades, una hoja por año) |
| **Salida** | `Datos/actividades_cleaned.csv` (68,159 filas) |
| **Funciones auxiliares** | [`utils.py`](utils.py): `normalizar_texto`, `find_different_activities`, `impute_new_registers` |
| **Siguiente paso** | [`../feature_extraction/feature_extraction.ipynb`](../feature_extraction/feature_extraction.ipynb) |

**Estrategia:** la planificación del curso cambió cada año (número de actividades, nombres y ponderaciones), por lo que limpiamos **año por año**, comparando contra la hoja correspondiente del Excel de actividades. Al final unimos los cuatro años y tratamos las fechas en conjunto.

### Resumen de decisiones

| # | Problema encontrado | Decisión | Justificación |
|---|---|---|---|
| 1 | Fechas almacenadas como texto | Convertir a `datetime` | Necesario para calcular entregas tardías y el mes de cada actividad |
| 2 | Actividades de estudiantes eliminados en la limpieza del registro | Conservar solo los `Est_ID` de `registro_cleaned.csv` | Sin registro no hay variable objetivo |
| 3 | Nombres de actividades con tildes y mayúsculas inconsistentes | Normalizar con `normalizar_texto` | Poder compararlos contra el Excel de planificación |
| 4 | Pares (estudiante, actividad) duplicados en todos los años | Eliminar duplicados conservando la primera ocurrencia | Cada estudiante realiza cada actividad una sola vez |
| 5 | 2020: 3 actividades con nombre distinto al del Excel (`FINAL`, `EVALUACION ACUMULATIVA`, `SIMULACRO 3 ACUMULATIVO`) | Renombrarlas a su equivalente del Excel | Tienen el mismo puntaje y un nombre casi idéntico |
| 6 | Estudiantes con actividades faltantes (sus puntos posibles no suman 100) | Imputarlas con `impute_new_registers`, repartiendo proporcionalmente los puntos que faltan para llegar a la nota final registrada | Mantener la consistencia entre la suma de las actividades y la nota final |
| 7 | 2020: una actividad sin nota de un estudiante que no se retiró | Asignarle nota 0 | Su nota ya cuadraba con la nota final: no entregó esa actividad |
| 8 | 2022: los nombres de las actividades no coinciden con el Excel | Conservar los nombres originales y validar que sumen 100 puntos por sección | La planificación cambió respecto al Excel |
| 9 | Columnas categóricas y fechas vacías en los registros imputados | Rellenar con el valor más frecuente de la misma (año, ciclo, actividad, sección) | Son características de la actividad, no del estudiante |
| 10 | `FechaEnvio` faltante | Estimarla con la mediana de días de diferencia respecto al vencimiento de esa actividad | Permite aproximar las entregas tardías |
| 11 | `RevisionPares` y `perteneceAzona` constantes | Eliminarlas | Una columna constante no aporta información |

## Análisis inicial

In [1]:
# Librerías y funciones auxiliares de limpieza
import pyreadr
import pandas as pd
import math
import seaborn as sns
import re
import unicodedata
from utils import *
import numpy as np

In [2]:
# Carga del registro de actividades exportado de Canvas
result = pyreadr.read_r('../../Datos/pensamiento.RData')
df_notas = result['pensamiento_final']

In [3]:
# Carga del registro limpio de estudiantes
df_registro = pd.read_csv('../../Datos/registro_cleaned.csv')

In [4]:
# Primeras filas del registro de actividades
df_notas.head()

,Est_ID,Actividad,Seccion,Ciclo,zona,Anio,TipoCalificacion,PuntosPosibles,RevisionPares,FechaTodoElDia,FechaVencimiento,TipoEntrega,Nota,FechaCalificacion,FechaEnvio,TipoEnvio,CantComentarios,NotaPtos,NotaPorciento,perteneceAzona
0,635.0,EVALUACIÓN ACUMULATIVA,10,1,True,2019,percent,14.0,false,None,None,on_paper,72.0,2019-04-01,None,NaN,0.0,10.08,72.0,True
1,1419.0,EVALUACIÓN ACUMULATIVA,10,1,True,2019,percent,14.0,false,None,None,on_paper,70.0,2019-04-01,None,NaN,0.0,9.80,70.0,True
2,781.0,EVALUACIÓN ACUMULATIVA,10,1,True,2019,percent,14.0,false,None,None,on_paper,96.0,2019-04-01,None,NaN,0.0,13.44,96.0,True
3,977.0,EVALUACIÓN ACUMULATIVA,10,1,True,2019,points,14.0,false,None,None,none,14.0,2019-04-05,None,NaN,0.0,14.00,100.0,True
4,853.0,EVALUACIÓN ACUMULATIVA,10,1,True,2019,percent,14.0,false,None,None,on_paper,75.0,2019-04-01,None,NaN,0.0,10.50,75.0,True


In [5]:
# Estadísticas descriptivas
df_notas.describe()

,Est_ID,PuntosPosibles,Nota,CantComentarios,NotaPtos,NotaPorciento
count,61943.000000,61943.000000,61837.0,61943.000000,61837.00,61836.0
mean,1353.907011,3.755743,inf,0.256187,inf,inf
std,785.755834,3.818276,NaN,0.602610,NaN,NaN
min,1.000000,0.000000,0.0,0.000000,0.00,0.0
25%,668.000000,2.000000,50.0,0.000000,0.90,65.0
50%,1333.000000,2.000000,85.0,0.000000,2.00,88.0
75%,2044.000000,3.000000,100.0,0.000000,2.58,100.0
max,2720.000000,15.000000,inf,20.000000,inf,inf


In [6]:
# Dimensiones del dataset
print(f'Filas {df_notas.shape[0]}')
print(f'Columnas {df_notas.shape[1]}')

Filas 61943
Columnas 20


In [7]:
# Dimensiones del dataset
print(f'Filas {df_notas.shape[0]}')
print(f'Columnas {df_notas.shape[1]}')

Filas 61943
Columnas 20


## Tipos de dato

Vamos a verificar que las columnas tengan un tipo de dato adecuado.

In [8]:
# Tipos de dato de cada columna
df_notas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61943 entries, 0 to 61942
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   Est_ID             61943 non-null  float64 
 1   Actividad          61943 non-null  object  
 2   Seccion            61943 non-null  object  
 3   Ciclo              61943 non-null  object  
 4   zona               61943 non-null  bool    
 5   Anio               61943 non-null  object  
 6   TipoCalificacion   61943 non-null  category
 7   PuntosPosibles     61943 non-null  float64 
 8   RevisionPares      61943 non-null  object  
 9   FechaTodoElDia     24862 non-null  object  
 10  FechaVencimiento   43988 non-null  object  
 11  TipoEntrega        61943 non-null  category
 12  Nota               61837 non-null  float64 
 13  FechaCalificacion  61943 non-null  object  
 14  FechaEnvio         39564 non-null  object  
 15  TipoEnvio          39564 non-null  category
 16  Cant

En general el tipo de dato de cada columna es bastante adecuado, solamente vamos a cambiar el tipo de dato de las fechas

In [9]:
# Conversión de las columnas de fecha a datetime
cols = ['FechaTodoElDia', 'FechaVencimiento', 'FechaCalificacion', 'FechaEnvio']
df_notas[cols] = df_notas[cols].apply(pd.to_datetime)

In [10]:
# Año como entero
df_notas['Anio'] = df_notas['Anio'].astype(int)

In [11]:
# Identificador del estudiante como texto
df_notas['Est_ID'] = df_notas['Est_ID'].astype('object')

Filtramos para conservar únicamente las actividades de los estudiantes que quedaron en el registro limpio. Sin su registro no tenemos la nota final ni las características de ingreso, así que no nos sirven para el modelo.

In [12]:
# Solo actividades de estudiantes del registro limpio
df_notas = df_notas[df_notas['Est_ID'].isin(df_registro['Est_ID'].unique())].copy()

In [13]:
# Dimensiones después del filtro
print(f'Filas {df_notas.shape[0]}')
print(f'Columnas {df_notas.shape[1]}')

Filas 56541
Columnas 20


## Normalización de cadenas de texto

Vamos a normalizar los nombres de las actividades disponibles en el dataset

In [14]:
# Normalización de los nombres de actividades
df_notas['Actividad'] = df_notas['Actividad'].apply(normalizar_texto)

## Validación de rangos

Vamos a verificar que las variables se encuentren en rangos adecuados. Empezamos por las notas

In [15]:
# Rangos de notas y puntos
df_notas[['Nota', 'PuntosPosibles', 'NotaPtos', 'NotaPorciento']].agg(['min', 'max'])


,Nota,PuntosPosibles,NotaPtos,NotaPorciento
min,0.0,0.5,0.0,0.0
max,100.0,15.0,15.0,100.0


`Nota` y `NotaPorciento` muestran la nota obtenida en porcentaje, por lo tanto es correcto que el valor mínimo sea 0 y el máximo 100.

`PuntosPosibles` y `NotaPtos` representan, respectivamente, el valor de la actividad dentro de la nota final (sobre 100 puntos) y los puntos que obtuvo el estudiante. Ninguna actividad vale más de 15 puntos y, como `NotaPorciento` no supera 100, nadie obtiene más puntos de los posibles: los rangos son válidos.

Ahora verificamos las fechas

In [16]:
# Rangos de fechas
df_notas[['FechaTodoElDia', 'FechaVencimiento', 'FechaCalificacion', 'FechaEnvio']].agg(['min', 'max'])

,FechaTodoElDia,FechaVencimiento,FechaCalificacion,FechaEnvio
min,2019-01-26,2019-01-26,2019-01-18,2020-01-23
max,2022-06-03,2022-06-05,2022-06-10,2022-06-09


Los rangos son correctos, pues el primer semestre de la universidad abarca los meses de enero y junio por lo general

Ahora verificamos los años

In [17]:
# Años disponibles
print(df_notas['Anio'].unique())

[2019 2020 2021 2022]


El conjunto de datos solo tiene datos de los años de 2019 a 2022, por lo tanto es correcto. 

In [18]:
# Ciclos disponibles
print(df_notas['Ciclo'].unique())

['1']


Iremos analizando las notas año con año e iremos haciendo las operaciones de limpieza necesarias en cada iteración. 

## Año 2019

Empezaremos limpiando las notas del año 2019

In [19]:
# Planificación oficial de actividades de 2019
activities_2019 = pd.read_excel('../../Datos/Actividades Pensamiento.xlsx', sheet_name='2019')

In [20]:
# Primeras actividades planificadas
activities_2019.head()

,Actividad,Puntos posibles
0,Verificación 1,2.0
1,Verificación 2,2.0
2,Verificación 3,2.0
3,Verificación 4,2.0
4,Verificación 5,2.0


In [21]:
# Columnas de la planificación
activities_2019.columns

Index(['Actividad', 'Puntos posibles'], dtype='object')

In [22]:
# Actividades registradas en 2019
notas_2019 = df_notas[df_notas['Anio'] == 2019].copy()

In [23]:
# Primeras filas de 2019
notas_2019.head()

,Est_ID,Actividad,Seccion,Ciclo,zona,Anio,TipoCalificacion,PuntosPosibles,RevisionPares,FechaTodoElDia,FechaVencimiento,TipoEntrega,Nota,FechaCalificacion,FechaEnvio,TipoEnvio,CantComentarios,NotaPtos,NotaPorciento,perteneceAzona
0,635.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,false,NaT,NaT,on_paper,72.0,2019-04-01,NaT,NaN,0.0,10.08,72.0,True
1,1419.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,false,NaT,NaT,on_paper,70.0,2019-04-01,NaT,NaN,0.0,9.80,70.0,True
2,781.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,false,NaT,NaT,on_paper,96.0,2019-04-01,NaT,NaN,0.0,13.44,96.0,True
4,853.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,false,NaT,NaT,on_paper,75.0,2019-04-01,NaT,NaN,0.0,10.50,75.0,True
5,160.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,false,NaT,NaT,on_paper,64.0,2019-04-22,NaT,NaN,0.0,8.96,64.0,True


Vemos que en la tabla de actividades, los nombres de estas están en mayúsculas, mientras que en el registro de actividades con minúsculas, vamos a convertir primero las del excel a mayúsculas

In [24]:
# Nombres originales de las actividades planificadas
activities_2019['Actividad']

0             Verificación 1 
1             Verificación 2 
2             Verificación 3 
3             Verificación 4 
4             Verificación 5 
5             Verificación 6 
6             Verificación 7 
7             Verificación 8 
8                Simulacro 1 
9                Simulacro 2 
10               Simulacro 3 
11               Simulacro 4 
12               Simulacro 5 
13          Trabajo Guiado 1 
14          Trabajo Guiado 2 
15                 Parcial 1 
16                 Parcial 2 
17                 Parcial 3 
18    Evaluación acumulativa 
19      Evaluación de cierre 
20              Examen Final 
21                        NaN
Name: Actividad, dtype: object

In [25]:
# Normalización de los nombres planificados
activities_2019['Actividad'] = activities_2019['Actividad'].apply(normalizar_texto)

In [26]:
# Nombres normalizados
activities_2019['Actividad']

0             VERIFICACION 1
1             VERIFICACION 2
2             VERIFICACION 3
3             VERIFICACION 4
4             VERIFICACION 5
5             VERIFICACION 6
6             VERIFICACION 7
7             VERIFICACION 8
8                SIMULACRO 1
9                SIMULACRO 2
10               SIMULACRO 3
11               SIMULACRO 4
12               SIMULACRO 5
13          TRABAJO GUIADO 1
14          TRABAJO GUIADO 2
15                 PARCIAL 1
16                 PARCIAL 2
17                 PARCIAL 3
18    EVALUACION ACUMULATIVA
19      EVALUACION DE CIERRE
20              EXAMEN FINAL
21                       NaN
Name: Actividad, dtype: object

In [27]:
# Actividades registradas en Canvas en 2019
notas_2019['Actividad'].unique()

array(['EVALUACION ACUMULATIVA', 'EVALUACION DE CIERRE', 'EXAMEN FINAL',
       'PARCIAL 1', 'PARCIAL 2', 'PARCIAL 3', 'SIMULACRO 1',
       'SIMULACRO 2', 'SIMULACRO 3', 'SIMULACRO 4', 'SIMULACRO 5',
       'TRABAJO GUIADO 1', 'TRABAJO GUIADO 2', 'VERIFICACION 1',
       'VERIFICACION 2', 'VERIFICACION 3', 'VERIFICACION 4',
       'VERIFICACION 5', 'VERIFICACION 6', 'VERIFICACION 7',
       'VERIFICACION 8'], dtype=object)

In [28]:
# Comparación entre la planificación y el registro
sorted(activities_2019['Actividad'][:21]) == sorted(notas_2019['Actividad'].unique())

True

Las actividades tienen el mismo nombre. Ahora veremos si suman 100

In [29]:
# Suma de puntos posibles planificados
print(activities_2019['Puntos posibles'][:21].sum())

100.0


In [30]:
# Actividades únicas con sus puntos posibles
suma_total = notas_2019[['Actividad', 'PuntosPosibles']].drop_duplicates().copy()

In [31]:
# Suma de puntos posibles registrados
print(suma_total['PuntosPosibles'].sum())

100.0


Las actividades suman 100 en ambos conjuntos. 

### Duplicados 

Ahora analizaremos si hay registros duplicados. Un registro duplicado será el par (id estudiante, actividad). Si existen duplicados, los eliminaremos conservando la primera ocurrencia (`keep='first'`).

In [32]:
# Registros por estudiante en 2019
notas_2019['Est_ID'].value_counts()

Est_ID
2566.0    42
1523.0    42
2240.0    42
390.0     42
2623.0    42
          ..
2129.0    21
2151.0    21
2695.0    21
300.0     21
30.0      21
Name: count, Length: 498, dtype: int64

Existen duplicados pues durante el semestre se realizaron 21 actividades y hay estudiantes que aparecen 42 veces, por lo tanto hay registros duplicados. 

In [33]:
# Eliminación de pares estudiante-actividad duplicados
notas_2019 = notas_2019.drop_duplicates(subset=['Est_ID', 'Actividad'], keep = 'first')

In [34]:
# Registros por estudiante sin duplicados
notas_2019['Est_ID'].value_counts()

Est_ID
2259.0    21
635.0     21
2020.0    21
2404.0    21
1133.0    21
          ..
1564.0    21
457.0     21
160.0     21
853.0     21
781.0     21
Name: count, Length: 498, dtype: int64

### Valores faltantes

Ahora analizamos si existen valores faltantes en el año 2019. El tratamiento de las columnas de las fechas lo dejaremos para después, por ahora nos vamos a centrar en la columna **Nota**

In [35]:
# Valores faltantes en 2019
notas_2019.isna().sum()

Est_ID                   0
Actividad                0
Seccion                  0
Ciclo                    0
zona                     0
Anio                     0
TipoCalificacion         0
PuntosPosibles           0
RevisionPares            0
FechaTodoElDia        9462
FechaVencimiento      9462
TipoEntrega              0
Nota                     0
FechaCalificacion        0
FechaEnvio           10458
TipoEnvio            10458
CantComentarios          0
NotaPtos                 0
NotaPorciento            0
perteneceAzona           0
dtype: int64

In [36]:
# Dimensiones de 2019
notas_2019.shape

(10458, 20)

Las fechas las vamos a tratar cuando hayamos analizado cada año de forma individual

## Año 2020

Ahora vamos a analizar las notas del año 2020

In [37]:
# Planificación oficial de actividades de 2020
activities_2020= pd.read_excel('../../Datos/Actividades Pensamiento.xlsx', sheet_name='2020')

In [38]:
# Primeras actividades planificadas
activities_2020.head()

,Actividad,Puntos posibles
0,Simulacro 1,2.4
1,Simulacro 2,2.4
2,Simulacro 3,2.4
3,Simulacro 4,2.4
4,Parcial 3,10.0


In [39]:
# Cantidad de actividades planificadas
activities_2020.shape

(35, 2)

In [40]:
# Actividades registradas en 2020
notas_2020 = df_notas[df_notas['Anio'] == 2020].copy()

In [41]:
# Dimensiones de 2020
notas_2020.shape

(8434, 20)

In [42]:
# Normalización de los nombres planificados
activities_2020['Actividad'] = activities_2020['Actividad'].apply(normalizar_texto)

Primero vamos a determinar si las actividades del registro de notas coinciden con las del archivo de actividades 

In [43]:
# Actividades registradas en Canvas en 2020
notas_2020['Actividad'].unique()

array(['EJERCITACION 1', 'EJERCITACION 10', 'EJERCITACION 11',
       'EJERCITACION 12', 'EJERCITACION 13', 'EJERCITACION 14',
       'EJERCITACION 2', 'EJERCITACION 3', 'EJERCITACION 4',
       'EJERCITACION 5', 'EJERCITACION 6', 'EJERCITACION 7',
       'EJERCITACION 8', 'EJERCITACION 9', 'EVALUACION ACUMULATIVA 1',
       'EVALUACION ACUMULATIVA', 'EVALUACION DE CIERRE', 'EXAMEN FINAL',
       'FINAL', 'PARCIAL 1', 'PARCIAL 2', 'PARCIAL 3', 'SIMULACRO 1',
       'SIMULACRO 2', 'SIMULACRO 3 ACUMULATIVO', 'SIMULACRO 3',
       'SIMULACRO 4', 'SIMULACRO 5', 'TRABAJO GUIADO 1',
       'TRABAJO GUIADO 2', 'VERIFICACION 1', 'VERIFICACION 2',
       'VERIFICACION 3', 'VERIFICACION 4', 'VERIFICACION 5',
       'VERIFICACION 6', 'VERIFICACION 7', 'VERIFICACION 8'], dtype=object)

In [44]:
# Actividades planificadas en 2020
activities_2020['Actividad']

0                  SIMULACRO 1
1                  SIMULACRO 2
2                  SIMULACRO 3
3                  SIMULACRO 4
4                    PARCIAL 3
5                  SIMULACRO 5
6                 EXAMEN FINAL
7                    PARCIAL 1
8                    PARCIAL 2
9     EVALUACION ACUMULATIVA 1
10              VERIFICACION 1
11        EVALUACION DE CIERRE
12              VERIFICACION 2
13              VERIFICACION 3
14              VERIFICACION 4
15              VERIFICACION 5
16              VERIFICACION 6
17              VERIFICACION 7
18              VERIFICACION 8
19            TRABAJO GUIADO 1
20            TRABAJO GUIADO 2
21              EJERCITACION 1
22              EJERCITACION 2
23              EJERCITACION 3
24              EJERCITACION 4
25              EJERCITACION 5
26              EJERCITACION 6
27              EJERCITACION 7
28              EJERCITACION 8
29              EJERCITACION 9
30             EJERCITACION 10
31             EJERCITACION 11
32      

In [45]:
# Suma de puntos posibles planificados
activities_2020['Puntos posibles'].sum()

np.float64(100.0)

In [46]:
# Comparación entre la planificación y el registro
sorted(activities_2020['Actividad']) == sorted(notas_2020['Actividad'].unique())

False

Los arrays no poseen las mismas actividades

In [47]:
# Cantidad de actividades planificadas
len(activities_2020['Actividad'])

35

In [48]:
# Cantidad de actividades registradas
len(notas_2020['Actividad'].unique())

38

En el registro de notas hay 3 actividades extras, vamos a verificar cuáles son

In [49]:
# Actividades registradas que no están en la planificación
diferentes = find_different_activities(notas_2020['Actividad'].unique(), activities_2020['Actividad'].values)

In [50]:
# Actividades distintas
print(diferentes)

['EVALUACION ACUMULATIVA', 'FINAL', 'SIMULACRO 3 ACUMULATIVO']


Las actividades que están en el registro de notas pero no en el de actividades son **Evaluación acumulativa**, **Final** y **Simulacro 3 acumulativo**. 

In [51]:
# Nombres planificados
activities_2020['Actividad'].values

array(['SIMULACRO 1', 'SIMULACRO 2', 'SIMULACRO 3', 'SIMULACRO 4',
       'PARCIAL 3', 'SIMULACRO 5', 'EXAMEN FINAL', 'PARCIAL 1',
       'PARCIAL 2', 'EVALUACION ACUMULATIVA 1', 'VERIFICACION 1',
       'EVALUACION DE CIERRE', 'VERIFICACION 2', 'VERIFICACION 3',
       'VERIFICACION 4', 'VERIFICACION 5', 'VERIFICACION 6',
       'VERIFICACION 7', 'VERIFICACION 8', 'TRABAJO GUIADO 1',
       'TRABAJO GUIADO 2', 'EJERCITACION 1', 'EJERCITACION 2',
       'EJERCITACION 3', 'EJERCITACION 4', 'EJERCITACION 5',
       'EJERCITACION 6', 'EJERCITACION 7', 'EJERCITACION 8',
       'EJERCITACION 9', 'EJERCITACION 10', 'EJERCITACION 11',
       'EJERCITACION 12', 'EJERCITACION 13', 'EJERCITACION 14'],
      dtype=object)

Son nombres bastantes parecidos a las actividades **Examen final**, **Evaluación acumulativa 1** y **Simulacro 3**, vamos a determinar si son las mismas actividades. 

In [52]:
# Puntos posibles de las actividades distintas
notas_2020[notas_2020['Actividad'].isin(diferentes)][['Actividad', 'PuntosPosibles']].drop_duplicates(subset='Actividad')

,Actividad,PuntosPosibles
13989,EVALUACION ACUMULATIVA,12.0
14733,FINAL,15.0
17200,SIMULACRO 3 ACUMULATIVO,2.4


In [53]:
# Puntos de las actividades planificadas equivalentes
activities_2020[activities_2020['Actividad'].isin(['EXAMEN FINAL', 'EVALUACION ACUMULATIVA 1', 'SIMULACRO 3'])]

,Actividad,Puntos posibles
2,SIMULACRO 3,2.4
6,EXAMEN FINAL,15.0
9,EVALUACION ACUMULATIVA 1,12.0


Las actividades poseen los mismos puntajes y nombres muy similares, por lo tanto vamos a cambiar el nombre de la actividad en el registro de notas. 

In [54]:
def clean_activity(valor):
    """Renombra las actividades de 2020 a su nombre en la planificación"""
    if valor == "SIMULACRO 3 ACUMULATIVO":
        return "SIMULACRO 3"
    elif valor == "FINAL":
        return "EXAMEN FINAL"
    elif valor == "EVALUACION ACUMULATIVA":
        return "EVALUACION ACUMULATIVA 1"
    return valor

# Aplicación del renombrado
notas_2020['Actividad'] = notas_2020['Actividad'].apply(clean_activity)

In [55]:
# Verificación de que ahora coinciden
sorted(activities_2020['Actividad']) == sorted(notas_2020['Actividad'].unique())

True

Ahora las actividades coinciden 

### Duplicados

Ahora vamos a determinar si hay registros duplicados. Cada estudiante no debe tener más de 35 registros

In [56]:
# Registros por estudiante en 2020
notas_2020['Est_ID'].value_counts()

Est_ID
1678.0    50
792.0     40
7.0       40
2680.0    36
842.0     34
          ..
1299.0     5
1518.0     5
721.0      4
1524.0     3
2691.0     3
Name: count, Length: 592, dtype: int64

In [57]:
# Eliminación de pares estudiante-actividad duplicados
notas_2020 = notas_2020.drop_duplicates(subset=['Est_ID', 'Actividad'], keep = 'first')

In [58]:
# Registros por estudiante sin duplicados
notas_2020['Est_ID'].value_counts()

Est_ID
1579.0    26
1070.0    25
1678.0    25
2314.0    25
658.0     25
          ..
1299.0     5
1518.0     5
721.0      4
1524.0     3
2691.0     3
Name: count, Length: 592, dtype: int64

Al borrar duplicados, observamos que los estudiantes tienen menos actividades de las que deberían, en el archivo excel de actividades existen 35 registradas para el 2020. Sin embargo, luego de eliminar duplicados, los estudiantes solamente tienen 26 actividades registradas. Vamos a verificar si la suma de los puntos posibles de estas actividades registradas suma 100

In [59]:
# Suma de puntos posibles por estudiante
notas_2020.groupby('Est_ID')['PuntosPosibles'].sum().sort_values(ascending=False)

Est_ID
1612.0    92.5
1100.0    91.5
912.0     91.1
710.0     90.5
1384.0    90.5
          ... 
1067.0    19.7
2457.0    16.7
1915.0    15.8
1524.0    14.9
1050.0    12.2
Name: PuntosPosibles, Length: 592, dtype: float64

Los puntos posibles de las actividades no suman 100, por lo tanto,  están incompletas, faltan puntos. 

Para rellenar las actividades faltantes, vamos a hacer lo siguiente:

- Filtramos para aquellos estudiantes que no se retiraron del curso, pues estos siempre van a tener actividades incompletas
- Recorremos uno a uno los estudiantes restantes
- Obtenemos del conjunto de registro, la nota final de un estudiante 
- En base a la nota final del estudiante registrada, calculamos cuantos puntos faltan para obtener la nota final
- Obtenemos las actividades faltantes del estudiante
- Calculamos cuántos puntos puede obtener el estudiante en base a las actividades faltantes
- Repartimos los puntos que faltan proporcionalmente 

La función para ejecutar dicho proceso se llama **impute_new_registers** y se encuentra en el archivo utils: [utils.py](utils.py)

> Los registros imputados quedan marcados con `imputado = True`. La columna `residuo_estudiante` guarda los puntos que no pudieron repartirse porque la nota de cada actividad se limita al rango [0, puntos posibles].

In [60]:
# Estudiantes en las actividades de 2020
len(notas_2020['Est_ID'].unique())

592

In [61]:
# Estudiantes en el registro de 2020
len(df_registro[df_registro['Anio'] == 2020]['Est_ID'].unique())

592

In [62]:
# Imputación de las actividades faltantes de cada estudiante
nuevos_registros = impute_new_registers(activities_2020, df_registro[df_registro['Anio'] == 2020], notas_2020)
notas_2020 = pd.concat([notas_2020, nuevos_registros])

Ahora verificamos si los estudiantes tienen sus actividades completas 

In [63]:
# Registros por estudiante después de imputar
notas_2020['Est_ID'].value_counts()

Est_ID
493.0     35
2427.0    35
2395.0    35
115.0     35
301.0     35
          ..
2266.0    35
1593.0    35
2296.0    35
1050.0    35
2457.0    35
Name: count, Length: 592, dtype: int64

In [64]:
# Verificación de que los puntos posibles suman 100
notas_2020.groupby('Est_ID')['PuntosPosibles'].sum()

Est_ID
5.0       100.0
7.0       100.0
8.0       100.0
12.0      100.0
16.0      100.0
          ...  
2700.0    100.0
2701.0    100.0
2704.0    100.0
2706.0    100.0
2711.0    100.0
Name: PuntosPosibles, Length: 592, dtype: float64

### Valores faltantes

Finalmente verificamos los valores faltantes

In [65]:
# Valores faltantes en 2020
notas_2020.isna().sum()

Est_ID                    0
Actividad                 0
Seccion                   0
Ciclo                     0
zona                      0
Anio                      0
TipoCalificacion          0
PuntosPosibles            0
RevisionPares             0
FechaTodoElDia         9589
FechaVencimiento       6508
TipoEntrega               0
Nota                      1
FechaCalificacion         0
FechaEnvio            17011
TipoEnvio              7167
CantComentarios           0
NotaPtos                  1
NotaPorciento             1
perteneceAzona            0
imputado               8201
residuo_estudiante     8201
dtype: int64

In [66]:
# Dimensiones de 2020
notas_2020.shape

(20720, 22)

Y ahora verificamos porque hay NA en las notas

In [67]:
# Registros sin nota
notas_2020[notas_2020[['Nota', 'NotaPorciento', 'NotaPtos']].isna().any(axis=1)]

,Est_ID,Actividad,Seccion,Ciclo,zona,Anio,TipoCalificacion,PuntosPosibles,RevisionPares,FechaTodoElDia,...,Nota,FechaCalificacion,FechaEnvio,TipoEnvio,CantComentarios,NotaPtos,NotaPorciento,perteneceAzona,imputado,residuo_estudiante
14283,758.0,EVALUACION DE CIERRE,80,1,True,2020,points,3.0,false,2020-06-01,...,NaN,2020-04-23,2020-04-23,online_quiz,0.0,NaN,NaN,False,NaN,NaN


Para ver la razón de porque tiene NA en ese campo, primero vamos a verificar si la nota que obtenemos al sumar todos los valores en NotaPtos es idéntica a la nota final del estudiante. 

In [68]:
# Comparación entre la suma de puntos y la nota final del estudiante 758
nota_sospechoso = notas_2020[notas_2020['Est_ID'] == 758.0]['NotaPtos'].sum()
nota_real = df_registro[df_registro['Est_ID'] == 758.0]['NotaFinal'].iloc[0]
print(f'Nota en registro {nota_real}, nota en base a calificaciones {nota_sospechoso}')

Nota en registro 83.0, nota en base a calificaciones 83.00000000000001


Verificamos si todos los puntos posibles de las actividades del estudiante suman 100

In [69]:
# Suma de puntos posibles del estudiante 758
print(notas_2020[notas_2020['Est_ID'] == 758.0]['PuntosPosibles'].sum())

100.0


Suman 100, por lo tanto, no vamos a eliminar el registro. Posiblemente el estudiante no entregó dicha actividad, vamos a ponerle nota = 0 y que sí pertenece a zona. 

In [70]:
# Nota 0 en la actividad no entregada
notas_2020.loc[14283, ['Nota', 'NotaPtos', 'NotaPorciento', 'perteneceAzona']] = [0.0, 0.0, 0.0, True]

## Año 2021

Ahora analizamos registros del año 2021

In [71]:
# Actividades registradas en 2021
notas_2021 = df_notas[df_notas['Anio'] == 2021].copy()

In [72]:
# Planificación oficial de actividades de 2021
activities_2021= pd.read_excel('../../Datos/Actividades Pensamiento.xlsx', sheet_name='2021')

In [73]:
# Actividades registradas en Canvas en 2021
notas_2021['Actividad'].unique()

array(['EJERCITACION 1', 'EJERCITACION 2', 'EJERCITACION 3',
       'EJERCITACION 4', 'EJERCITACION 5', 'EJERCITACION 6',
       'EJERCITACION 7', 'EVALUACION ACUMULATIVA', 'EVALUACION DE CIERRE',
       'EXAMEN FINAL', 'PARCIAL 1', 'PARCIAL 2', 'PARCIAL 3',
       'SIMULACRO 1', 'SIMULACRO 2', 'SIMULACRO 3', 'SIMULACRO 4',
       'SIMULACRO 5', 'TAREA 1', 'TAREA 10', 'TAREA 11', 'TAREA 12',
       'TAREA 13', 'TAREA 14', 'TAREA 2', 'TAREA 3', 'TAREA 4', 'TAREA 5',
       'TAREA 6', 'TAREA 7', 'TAREA 8', 'TAREA 9', 'TRABAJO GUIADO 1',
       'TRABAJO GUIADO 2'], dtype=object)

In [74]:
# Normalización de los nombres planificados
activities_2021['Actividad'] = activities_2021['Actividad'].apply(normalizar_texto)

Verificamos si la cantidad de actividades coincide

In [75]:
# Cantidad de actividades planificadas
len(activities_2021['Actividad'])

34

In [76]:
# Cantidad de actividades registradas
len(notas_2021['Actividad'].unique())

34

Y ahora verificamos que los nombres de las actividades coinciden

In [77]:
# Comparación entre la planificación y el registro
sorted(activities_2021['Actividad']) == sorted(notas_2021['Actividad'].unique())

True

El número y nombre de las actividades coincide.

## Duplicados

Vamos a determinar si hay registros duplicados.

In [78]:
# Registros por estudiante en 2021
notas_2021['Est_ID'].value_counts()

Est_ID
515.0     68
1626.0    68
412.0     68
612.0     68
2299.0    68
          ..
2327.0    34
872.0     34
399.0     34
1019.0    34
244.0     34
Name: count, Length: 613, dtype: int64

Hay estudiantes con 68 registros, el doble de las actividades registradas, los vamos a eliminar. 

In [79]:
# Eliminación de pares estudiante-actividad duplicados
notas_2021 = notas_2021.drop_duplicates(subset=['Est_ID', 'Actividad'], keep='first')

In [80]:
# Registros por estudiante sin duplicados
notas_2021['Est_ID'].value_counts()

Est_ID
1318.0    34
2385.0    34
295.0     34
284.0     34
82.0      34
          ..
2600.0    34
857.0     34
207.0     34
2352.0    34
1523.0    20
Name: count, Length: 613, dtype: int64

In [81]:
# Nota final registrada del estudiante 1523
df_registro[(df_registro['Est_ID'] == 1523) & (df_registro['Anio'] == 2021)]

,Est_ID,PAA,Colegio,TituloNivelMedio,NotaFinal,Aprobado,Genero,EstadoCurso,veces_curso,Anio
2296,1523.0,992.0,COLEGIO CIUDAD VIEJA,BACHILLER EN CIENCIAS Y LETRAS,9.0,0,MASCULINO,REPROBADO,2,2021


In [82]:
# Suma de puntos del estudiante 1523
notas_2021[notas_2021['Est_ID'] == 1523]['NotaPtos'].sum()

np.float64(8.385)

La nota coincide con la que existe en registro. 

In [83]:
# Imputación de las actividades faltantes del estudiante 1523
suspicious_student = notas_2021[notas_2021['Est_ID'] == 1523.0]
nuevos_registros_2021 = impute_new_registers(activities_2021, df_registro[df_registro['Anio'] == 2021], suspicious_student)

In [84]:
# Unión de las actividades imputadas
notas_2021 = pd.concat([notas_2021, nuevos_registros_2021])

In [85]:
# Registros por estudiante después de imputar
notas_2021['Est_ID'].value_counts()

Est_ID
1318.0    34
2385.0    34
295.0     34
284.0     34
82.0      34
          ..
2708.0    34
2600.0    34
857.0     34
207.0     34
2352.0    34
Name: count, Length: 613, dtype: int64

In [86]:
# Verificación de la nota del estudiante 1523
notas_2021[notas_2021['Est_ID'] == 1523]['NotaPtos'].sum()

np.float64(9.0)

## Valores faltantes

Verificamos si hay valores faltantes en las columnas relacionadas a la nota

In [87]:
# Valores faltantes en 2021
notas_2021.isna().sum()

Est_ID                    0
Actividad                 0
Seccion                   0
Ciclo                     0
zona                      0
Anio                      0
TipoCalificacion         14
PuntosPosibles            0
RevisionPares            14
FechaTodoElDia        12749
FechaVencimiento        376
TipoEntrega              14
Nota                      0
FechaCalificacion        14
FechaEnvio             1540
TipoEnvio              1540
CantComentarios           0
NotaPtos                  0
NotaPorciento             0
perteneceAzona            0
imputado              20828
residuo_estudiante    20828
dtype: int64

In [88]:
# Dimensiones de 2021
notas_2021.shape

(20842, 22)

No hay NA relevantes en las columnas de notas. Los NA en `TipoCalificacion`, `RevisionPares` y `TipoEntrega` corresponden a los registros imputados y se rellenan al final del notebook.

## Año 2022

Ahora analizaremos las notas del año 2022

In [89]:
# Actividades registradas en 2022
notas_2022 = df_notas[df_notas['Anio'] == 2022].copy()

In [90]:
# Planificación oficial de actividades de 2022
activities_2022 = pd.read_excel('../../Datos/Actividades Pensamiento.xlsx', sheet_name='2022')

In [91]:
# Normalización de los nombres planificados
activities_2022['Actividad'] = activities_2022['Actividad'].apply(normalizar_texto)

Verificamos si existen la misma cantidad de actividades

In [92]:
# Cantidad de actividades planificadas
len(activities_2022['Actividad'])

34

In [93]:
# Cantidad de actividades registradas
len(notas_2022['Actividad'].unique())

27

In [94]:
# Comparación entre la planificación y el registro
sorted(activities_2022['Actividad']) == sorted(notas_2022['Actividad'].unique())

False

El registro de actividades y el de notas no coinciden

In [95]:
# Actividades planificadas
activities_2022['Actividad'].unique()

array(['ACTIVIDAD ASINCRONICA 1', 'ACTIVIDAD ASINCRONICA 2',
       'ACTIVIDAD ASINCRONICA 3', 'EJERCITACION 1', 'EJERCITACION 10',
       'EJERCITACION 2', 'EJERCITACION 3', 'EJERCITACION 4',
       'EJERCITACION 5', 'EJERCITACION 6', 'EJERCITACION 7',
       'EJERCITACION 8', 'EJERCITACION 9', 'PARCIAL 1', 'PARCIAL 2',
       'PARCIAL 3', 'PARCIAL 4', 'SIMULACRO 1', 'SIMULACRO 2',
       'SIMULACRO 3', 'TAREA 1', 'TAREA 10', 'TAREA 11', 'TAREA 12',
       'TAREA 13', 'TAREA 14', 'TAREA 2', 'TAREA 3', 'TAREA 4', 'TAREA 5',
       'TAREA 6', 'TAREA 7', 'TAREA 8', 'TAREA 9'], dtype=object)

In [96]:
# Actividades registradas
notas_2022['Actividad'].unique()

array(['ACTIVIDAD INTEGRADORA 1 TRANSFORMACION DE FUNCIONES',
       'ACTIVIDAD INTEGRADORA 2 GRAFFUNPOL YRAC',
       'ACTIVIDAD INTEGRADORA 3 FUNCIONES EXPONENCIALES Y LOGARITMICAS',
       'ACTIVIDAD INTEGRADORA 4 ECTRIG', 'AUTOAPRENDIZAJE 1',
       'AUTOAPRENDIZAJE 2', 'AUTOAPRENDIZAJE 3 PARTE 2',
       'AUTOAPRENDIZAJE 3', 'AUTOAPRENDIZAJE 4', 'EJERCITACION 1',
       'EJERCITACION 2', 'EJERCITACION 3', 'EJERCITACION 4',
       'EJERCITACION 5', 'EJERCITACION 6', 'EJERCITACION 7',
       'EJERCITACION 8', 'EJERCITACION 9', 'EVALUACION DE CIERRE',
       'PARCIAL 1', 'PARCIAL 2', 'PARCIAL 3', 'PARCIAL 4', 'SIMULACRO 1',
       'SIMULACRO 2', 'SIMULACRO 3', 'SIMULACRO 4'], dtype=object)

Los nombres de las actividades no coinciden, son totalmente diferentes. Vamos a mantener el nombre original de cada actividad, simplemente vamos a verificar si el puntaje total de todas suma 100 en todas las secciones. También verificaremos si todas las secciones realizaron las mismas actividades. 

In [97]:
# Actividades realizadas por sección
resultado = notas_2022.groupby('Seccion')['Actividad'].unique()

In [98]:
# Cantidad de actividades por sección
longitudes = resultado.apply(len)

In [99]:
# Resultado por sección
longitudes

Seccion
10     26
100    26
110    26
120    25
130    26
140    26
150    26
160    26
170    26
180    26
190    26
20     26
200    26
210    26
30     26
40     26
50     26
60     26
70     26
80     26
90     26
Name: Actividad, dtype: int64

Solamente en la sección 120 se realizaron 25 actividades.  Veremos que actividad extra se realizó en las demás secciones

In [100]:
# Actividades de las demás secciones que no se hicieron en la sección 120
print(find_different_activities(notas_2022[notas_2022['Seccion'] != '120']['Actividad'].unique(),
                          notas_2022[notas_2022['Seccion'] == '120']['Actividad'].unique()))

['AUTOAPRENDIZAJE 3', 'AUTOAPRENDIZAJE 4']


In [101]:
# Actividades de las secciones distintas a la 120
notas_2022[notas_2022['Seccion'] != '120']['Actividad'].unique()

array(['ACTIVIDAD INTEGRADORA 1 TRANSFORMACION DE FUNCIONES',
       'ACTIVIDAD INTEGRADORA 2 GRAFFUNPOL YRAC',
       'ACTIVIDAD INTEGRADORA 3 FUNCIONES EXPONENCIALES Y LOGARITMICAS',
       'ACTIVIDAD INTEGRADORA 4 ECTRIG', 'AUTOAPRENDIZAJE 1',
       'AUTOAPRENDIZAJE 2', 'AUTOAPRENDIZAJE 3', 'AUTOAPRENDIZAJE 4',
       'EJERCITACION 1', 'EJERCITACION 2', 'EJERCITACION 3',
       'EJERCITACION 4', 'EJERCITACION 5', 'EJERCITACION 6',
       'EJERCITACION 7', 'EJERCITACION 8', 'EJERCITACION 9',
       'EVALUACION DE CIERRE', 'PARCIAL 1', 'PARCIAL 2', 'PARCIAL 3',
       'PARCIAL 4', 'SIMULACRO 1', 'SIMULACRO 2', 'SIMULACRO 3',
       'SIMULACRO 4'], dtype=object)

In [102]:
# Actividades de la sección 120
notas_2022[notas_2022['Seccion'] == '120']['Actividad'].unique()

array(['ACTIVIDAD INTEGRADORA 1 TRANSFORMACION DE FUNCIONES',
       'ACTIVIDAD INTEGRADORA 2 GRAFFUNPOL YRAC',
       'ACTIVIDAD INTEGRADORA 3 FUNCIONES EXPONENCIALES Y LOGARITMICAS',
       'ACTIVIDAD INTEGRADORA 4 ECTRIG', 'AUTOAPRENDIZAJE 1',
       'AUTOAPRENDIZAJE 2', 'AUTOAPRENDIZAJE 3 PARTE 2', 'EJERCITACION 1',
       'EJERCITACION 2', 'EJERCITACION 3', 'EJERCITACION 4',
       'EJERCITACION 5', 'EJERCITACION 6', 'EJERCITACION 7',
       'EJERCITACION 8', 'EJERCITACION 9', 'EVALUACION DE CIERRE',
       'PARCIAL 1', 'PARCIAL 2', 'PARCIAL 3', 'PARCIAL 4', 'SIMULACRO 1',
       'SIMULACRO 2', 'SIMULACRO 3', 'SIMULACRO 4'], dtype=object)

In [103]:
# Actividades únicas por sección con sus puntos posibles
suma_2022 = notas_2022.drop_duplicates(subset=['Seccion', 'Actividad', 'PuntosPosibles'], keep = 'first')

In [104]:
# Suma de puntos posibles por sección
for i in suma_2022['Seccion'].unique():
    print(suma_2022[suma_2022['Seccion'] == i]['PuntosPosibles'].sum())

100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0
100.0


Las actividades suman 100 puntos a pesar de no ser exactamente las mismas en todas las secciones. 

## Duplicados

Ahora analizamos los duplicados 

In [105]:
# Registros por estudiante en 2022
notas_2022['Est_ID'].value_counts()

Est_ID
2301.0    52
481.0     52
246.0     52
439.0     52
342.0     52
          ..
894.0     25
406.0     25
422.0     17
2539.0    12
722.0     10
Name: count, Length: 622, dtype: int64

Hay estudiantes con actividades duplicadas, eliminaremos los registros. 

In [106]:
# Eliminación de pares estudiante-actividad duplicados
notas_2022 = notas_2022.drop_duplicates(subset=['Est_ID', 'Actividad'], keep='first')

In [107]:
# Registros por estudiante sin duplicados
print(notas_2022['Est_ID'].value_counts())

Est_ID
2203.0    26
901.0     26
696.0     26
2.0       26
202.0     26
          ..
894.0     25
406.0     25
422.0     17
2539.0    12
722.0     10
Name: count, Length: 622, dtype: int64


Aún así podemos observar que hay estudiantes con menos actividades de las que se realizaron en cada sección. Primero vamos a verificar que los estudiantes de la sección 120 tengan 25 actividades y no menos para determinar bien nuestros filtros. 

In [108]:
# Registros de los estudiantes de la sección 120
print(notas_2022[notas_2022['Seccion'] == '120']['Est_ID'].value_counts())

Est_ID
349.0     25
783.0     25
543.0     25
1399.0    25
894.0     25
2399.0    25
1876.0    25
406.0     25
370.0     25
427.0     25
1020.0    25
682.0     25
1692.0    25
2572.0    25
2216.0    25
1752.0    25
629.0     25
1837.0    25
2631.0    25
2015.0    25
1053.0    25
1204.0    25
1483.0    25
545.0     25
219.0     25
2154.0    25
2649.0    25
2288.0    25
1167.0    25
581.0     25
124.0     25
2615.0    25
1561.0    25
Name: count, dtype: int64


Los estudiantes de dicha sección si tienen sus actividades completas. 

In [109]:
# Estudiantes de la sección 120
seccion_120_22_ids = notas_2022[notas_2022['Seccion'] == '120']['Est_ID'].unique()

Filtramos para determinar que estudiantes tienen menos actividades. 

In [110]:
# Estudiantes de las demás secciones con menos actividades
cantidad_actividades_22 = notas_2022['Est_ID'].value_counts()
cantidad_actividades_22 = cantidad_actividades_22[~cantidad_actividades_22.index.isin(seccion_120_22_ids)]
print(cantidad_actividades_22)


Est_ID
2203.0    26
901.0     26
696.0     26
2.0       26
202.0     26
          ..
360.0     26
1994.0    26
422.0     17
2539.0    12
722.0     10
Name: count, Length: 589, dtype: int64


In [111]:
# Puntos posibles del estudiante 422
print(notas_2022[notas_2022['Est_ID'] == 422.0]['PuntosPosibles'].sum())

59.5


In [112]:
# Puntos posibles del estudiante 2539
print(notas_2022[notas_2022['Est_ID'] == 2539.0]['PuntosPosibles'].sum())

47.0


In [113]:
# Puntos posibles del estudiante 722
print(notas_2022[notas_2022['Est_ID'] == 722.0]['PuntosPosibles'].sum())

42.5


In [114]:
# Nota final registrada de los tres estudiantes
print(df_registro[(df_registro['Est_ID'].isin([422.0, 2539.0, 722.0 ])) & (df_registro['Anio'] == 2022)][['Est_ID', 'NotaFinal']])

      Est_ID  NotaFinal
2262   422.0       14.0
2278   722.0        9.0
2318  2539.0        2.0


In [115]:
# Suma de puntos obtenidos de los tres estudiantes
print(notas_2022[notas_2022['Est_ID'] == 722.0]['NotaPtos'].sum())
print(notas_2022[notas_2022['Est_ID'] == 422.0]['NotaPtos'].sum())
print(notas_2022[notas_2022['Est_ID'] == 2539.0]['NotaPtos'].sum())

8.610000000000001
13.309999999999999
2.0


Faltan actividades, así que usamos la función `impute_new_registers` para completarlas. Como los nombres de 2022 no coinciden con el Excel, construimos la lista de actividades de referencia a partir de las secciones distintas a la 120, que tienen las 26 actividades.

In [116]:
# Lista de actividades de referencia a partir de las secciones distintas a la 120
activities_2022_fixed = (
    notas_2022[notas_2022['Seccion'] != '120']
    [['Actividad', 'PuntosPosibles']]
    .drop_duplicates(subset='Actividad')
    .reset_index(drop=True)
)

In [117]:
# Mismo nombre de columna que la planificación
activities_2022_fixed = activities_2022_fixed.rename(columns={'PuntosPosibles': 'Puntos posibles'})

In [118]:
# Actividades de los estudiantes incompletos
suspicious_student_22 = notas_2022[notas_2022['Est_ID'].isin([422.0, 2539.0, 722.0])]

In [119]:
# Imputación de sus actividades faltantes
nuevos_registros_2022 = impute_new_registers(activities_2022_fixed, df_registro[df_registro['Anio'] == 2022], suspicious_student_22)

In [120]:
# Unión de las actividades imputadas
notas_2022= pd.concat([notas_2022, nuevos_registros_2022])

In [121]:
# Registros por estudiante después de imputar
print(notas_2022['Est_ID'].value_counts())

Est_ID
2203.0    26
901.0     26
696.0     26
2.0       26
202.0     26
          ..
1399.0    25
894.0     25
2399.0    25
1876.0    25
406.0     25
Name: count, Length: 622, dtype: int64


## Valores faltantes

Verificamos por valores faltantes 

In [122]:
# Valores faltantes en 2022
notas_2022.isna().sum()

Est_ID                    0
Actividad                 0
Seccion                   0
Ciclo                     0
zona                      0
Anio                      0
TipoCalificacion         27
PuntosPosibles            0
RevisionPares            27
FechaTodoElDia         6830
FechaVencimiento       1845
TipoEntrega              27
Nota                      0
FechaCalificacion        27
FechaEnvio             2564
TipoEnvio              2564
CantComentarios           0
NotaPtos                  0
NotaPorciento             0
perteneceAzona            0
imputado              16100
residuo_estudiante    16100
dtype: int64

In [123]:
# Dimensiones de 2022
notas_2022.shape

(16139, 22)

## Operaciones finales

Juntamos las notas de todos los años

In [124]:
# Unión de los cuatro años
notas_clean = pd.concat(
    [notas_2019, notas_2020, notas_2021, notas_2022],
    ignore_index=True
)

In [125]:
# Dimensiones del dataset completo
notas_clean.shape

(68159, 22)

Introdujimos registros nuevos, por esa razón vimos algunos NA en varias columnas en los dataset de todos los años, por ejemplo, vimos que hay NA en TipoEntrega, revisionPares, etc. Vamos a rellenar esos registros con información de otros registros. Para ello, vamos a agrupar por año, sección, ciclo y actividad, pues son características intrínsecas de una actividad y cada sección en diferentes años y ciclos tiene valores diferentes para estas columnas. Esto podrá ser de ayuda para obtener features de rendimiento del estudiante como la cantidad de actividades entregadas tarde. 

 Antes de empezar, vamos a verificar la razón de porque en el año 2019 hay tantos NA en las columnas relacionadas a la fecha. 

In [126]:
# Tipos de entrega en 2019 (el filtro compara con texto, por eso los conteos salen en 0)
notas_clean[notas_clean['Anio'] == '2019']['TipoEntrega'].value_counts()

TipoEntrega
discussion_topic                                              0
external_tool                                                 0
media_recording,online_upload                                 0
none                                                          0
not_graded                                                    0
on_paper                                                      0
online_quiz                                                   0
online_text_entry                                             0
online_text_entry,online_upload                               0
online_text_entry,online_url,media_recording,online_upload    0
online_text_entry,online_url,online_upload                    0
online_upload                                                 0
online_url,media_recording,online_upload                      0
online_url,online_upload                                      0
Name: count, dtype: int64

En los registros de 2019 predominan las entregas en papel (`on_paper`, ver el `head` al inicio del notebook), posiblemente porque aún no se había adoptado por completo el LMS Canvas. Por eso casi no hay fechas de vencimiento ni de envío ese año, y no es posible calcular entregas tardías para 2019.

> **Nota:** la celda anterior compara `Anio` con el texto `'2019'`, pero en este punto la columna es entera; por eso todos los conteos salen en 0. El filtro correcto sería `notas_clean['Anio'] == 2019`.

In [127]:
# Estudiantes por año
estudiantes_por_año = notas_clean.groupby('Anio')['Est_ID'].nunique()

print(estudiantes_por_año)

Anio
2019    498
2020    592
2021    613
2022    622
Name: Est_ID, dtype: int64


In [128]:
# Porcentaje de estudiantes de 2019
print(round(((estudiantes_por_año[2019]/estudiantes_por_año.sum())) * 100, 2))

21.42


La mayoría de los estudiantes pertenecen a años posteriores al 2019 (casi el 80%). Por lo tanto, podemos permitir la falta de información sobre entregas fuera de tiempo en dichos registros. 

In [129]:
# Valores faltantes del dataset completo
notas_clean.isna().sum()

Est_ID                    0
Actividad                 0
Seccion                   0
Ciclo                     0
zona                      0
Anio                      0
TipoCalificacion         41
PuntosPosibles            0
RevisionPares            41
FechaTodoElDia        38630
FechaVencimiento      18191
TipoEntrega              41
Nota                      0
FechaCalificacion        41
FechaEnvio            31573
TipoEnvio             21729
CantComentarios           0
NotaPtos                  0
NotaPorciento             0
perteneceAzona            0
imputado              55587
residuo_estudiante    55587
dtype: int64

In [130]:
# Dimensiones del dataset completo
notas_clean.shape

(68159, 22)

Vamos a rellenar los valores faltantes a partir de otros registros de la misma actividad. La celda siguiente hace lo siguiente:

1. **Columnas de la actividad** (`TipoEntrega`, `TipoCalificacion`, `TipoEnvio`, `FechaTodoElDia`, `FechaVencimiento`, `FechaCalificacion`): se rellenan con el valor más frecuente de la misma combinación (año, ciclo, actividad, sección). `FechaVencimiento` no se rellena para parciales, exámenes, evaluaciones ni entregas en papel.
2. **`FechaEnvio`:** se rellena con el valor más frecuente de la misma actividad y sección.
3. Si falta `FechaVencimiento` pero existe `FechaTodoElDia`, se usa esta última; si `FechaVencimiento` es posterior a `FechaTodoElDia`, se limita a `FechaTodoElDia`.
4. Las `FechaEnvio` que aún falten se estiman como `FechaVencimiento` + la **mediana** de días de diferencia (envío − vencimiento) de esa actividad. Usamos la mediana porque es robusta a entregas muy tempranas o muy tardías.

Los parciales, exámenes y evaluaciones se excluyen de los pasos 2 a 4 porque se rinden en una fecha fija y no tienen plazo de entrega.

In [131]:
# Relleno de valores faltantes a partir de otros registros de la misma actividad
def valor_mas_frecuente(serie):
    """Devuelve el valor más frecuente de una serie, ignorando NaN/NaT.
    Si no hay valores válidos, devuelve NaN."""
    serie_valida = serie.dropna()
    if serie_valida.empty:
        return np.nan
    return serie_valida.mode().iloc[0]

columnas_categoricas = ['TipoEntrega', 'TipoCalificacion', 'TipoEnvio']
columnas_fecha = ['FechaTodoElDia', 'FechaVencimiento', 'FechaCalificacion']
todas_las_columnas = columnas_categoricas + columnas_fecha

# Máscara reutilizable: la actividad NO es parcial / examen / evaluación
mask_no_especial = (
    ~notas_clean['Actividad'].str.contains('PARCIAL', na=False)
    & ~notas_clean['Actividad'].str.contains('EXAMEN', na=False)
    & ~notas_clean['Actividad'].str.contains('EVALUACION', na=False)
)

# Tabla lookup: una fila por (Anio, Ciclo, Actividad, Seccion)
lookup_tabla = notas_clean.groupby(
    ['Anio', 'Ciclo', 'Actividad', 'Seccion']
)[todas_las_columnas].agg(valor_mas_frecuente)

# Índice de 4 niveles reutilizable (Anio, Ciclo, Actividad, Seccion)
idx_df = pd.MultiIndex.from_arrays([
    notas_clean['Anio'], notas_clean['Ciclo'],
    notas_clean['Actividad'], notas_clean['Seccion']
])

# Índice de 3 niveles (Anio, Ciclo, Actividad) para el offset por actividad
idx_df_actividad = pd.MultiIndex.from_arrays([
    notas_clean['Anio'], notas_clean['Ciclo'], notas_clean['Actividad']
])

# Columnas de la actividad con el valor más frecuente de su grupo
for col in todas_las_columnas:
    valores_calculados = pd.Series(idx_df.map(lookup_tabla[col]), index=notas_clean.index)

    if col == 'FechaVencimiento':
        mask = (
            notas_clean[col].isna()
            & mask_no_especial
            & (notas_clean['TipoEntrega'] != 'on_paper')
        )
    else:
        mask = notas_clean[col].isna()

    notas_clean[col] = notas_clean[col].mask(mask, valores_calculados)

# Paso 1: FechaEnvio con el valor más frecuente de la misma actividad y sección
lookup_envio = notas_clean.groupby(
    ['Anio', 'Ciclo', 'Actividad', 'Seccion']
)['FechaEnvio'].agg(valor_mas_frecuente)

valores_envio_seccion = pd.Series(idx_df.map(lookup_envio), index=notas_clean.index)

mask_envio_seccion = notas_clean['FechaEnvio'].isna() & mask_no_especial
notas_clean['FechaEnvio'] = notas_clean['FechaEnvio'].mask(mask_envio_seccion, valores_envio_seccion)

# Paso 2: si falta FechaVencimiento pero existe FechaTodoElDia, se usa FechaTodoElDia
mask_venc_desde_dia = (
    notas_clean['FechaVencimiento'].isna()
    & notas_clean['FechaTodoElDia'].notna()
    & mask_no_especial
)
notas_clean['FechaVencimiento'] = notas_clean['FechaVencimiento'].mask(
    mask_venc_desde_dia, notas_clean['FechaTodoElDia']
)

# Paso 3: si FechaVencimiento es posterior a FechaTodoElDia, se limita a FechaTodoElDia
mask_venc_tope = (
    notas_clean['FechaEnvio'].notna()
    & notas_clean['FechaVencimiento'].notna()
    & notas_clean['FechaTodoElDia'].notna()
    & (notas_clean['FechaVencimiento'] > notas_clean['FechaTodoElDia'])
    & mask_no_especial
)
notas_clean['FechaVencimiento'] = notas_clean['FechaVencimiento'].mask(
    mask_venc_tope, notas_clean['FechaTodoElDia']
)

# Paso 4: FechaEnvio restante estimada con la mediana de días de diferencia de la actividad
notas_clean['dias_diferencia'] = (
    notas_clean['FechaEnvio'] - notas_clean['FechaVencimiento']
).dt.days

offset_actividad = (
    notas_clean.loc[mask_no_especial]
    .groupby(['Anio', 'Ciclo', 'Actividad'])['dias_diferencia']
    .median()
)
offset_calculado = pd.Series(idx_df_actividad.map(offset_actividad), index=notas_clean.index)

fecha_envio_estimada = notas_clean['FechaVencimiento'] + pd.to_timedelta(offset_calculado, unit='D')

mask_envio_offset = notas_clean['FechaEnvio'].isna() & mask_no_especial
notas_clean['FechaEnvio'] = notas_clean['FechaEnvio'].mask(mask_envio_offset, fecha_envio_estimada)

Creamos la variable **`entrega_tarde`** (1 si `FechaEnvio` > `FechaVencimiento`). Excluimos parciales, exámenes y evaluaciones porque no son entregables con plazo: se rinden en una fecha fija.

In [132]:
# Entrega tardía: envío posterior al vencimiento, sin contar parciales, exámenes ni evaluaciones
notas_clean['entrega_tarde'] = (
    (notas_clean['FechaEnvio'] > notas_clean['FechaVencimiento'])
    &
    ~notas_clean['Actividad'].str.contains(
        'PARCIAL|EXAMEN|EVALUACION',
        case=False,
        na=False
    )
).astype(int)

In [133]:
# Valores faltantes después del relleno
notas_clean.isna().sum()

Est_ID                    0
Actividad                 0
Seccion                   0
Ciclo                     0
zona                      0
Anio                      0
TipoCalificacion          0
PuntosPosibles            0
RevisionPares            41
FechaTodoElDia        37311
FechaVencimiento      16800
TipoEntrega               0
Nota                      0
FechaCalificacion         0
FechaEnvio            19199
TipoEnvio             17179
CantComentarios           0
NotaPtos                  0
NotaPorciento             0
perteneceAzona            0
imputado              55587
residuo_estudiante    55587
dias_diferencia       23840
entrega_tarde             0
dtype: int64

Logramos rellenar gran parte de los registros relacionados con la fecha. 

Al principio hicimos una revisión inicial del head del dataset y vimos que revisión pares casi siempre es false. Al analizar registros por año, también vimos que el valor en esta columna por lo regular era false. Si siempre es false no aporta información relevante, podemos eliminar esa columna. 

In [134]:
# Valores de RevisionPares
notas_clean['RevisionPares'].value_counts()

RevisionPares
false    68118
Name: count, dtype: int64

Otra columna que parece tener un valor constante es pertenece a zona, por lo regular siempre es True, vamos a verificar. 

In [135]:
# Valores de perteneceAzona
notas_clean['perteneceAzona'].value_counts()

perteneceAzona
True    68159
Name: count, dtype: int64

Como ambas columnas son constantes, vamos a eliminarlas

In [136]:
# Eliminación de columnas constantes y auxiliares
notas_clean = notas_clean.drop(columns=['RevisionPares', 'perteneceAzona', 'dias_diferencia'])

Finalizamos el proceso de limpieza, guardamos el conjunto de datos. 

In [137]:
# Exportación de las actividades limpias
notas_clean.to_csv('../../Datos/actividades_cleaned.csv', index=False)

### Verificaciones finales
Comprobamos que ninguna actividad tenga `NotaPorciento` mayor a 100 (el resultado es un arreglo vacío).

In [138]:
# Actividades con porcentaje mayor a 100
notas_clean[notas_clean['NotaPorciento'] > 100]['Est_ID'].unique()

array([], dtype=object)

In [139]:
# Estudiantes con porcentaje mayor a 100
df_registro[df_registro['Est_ID'].isin(notas_clean[notas_clean['NotaPorciento'] > 100]['Est_ID'].unique())]

,Est_ID,PAA,Colegio,TituloNivelMedio,NotaFinal,Aprobado,Genero,EstadoCurso,veces_curso,Anio


In [140]:
# Dataset final
notas_clean

,Est_ID,Actividad,Seccion,Ciclo,zona,Anio,TipoCalificacion,PuntosPosibles,FechaTodoElDia,FechaVencimiento,...,Nota,FechaCalificacion,FechaEnvio,TipoEnvio,CantComentarios,NotaPtos,NotaPorciento,imputado,residuo_estudiante,entrega_tarde
0,635.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,NaT,NaT,...,72.000000,2019-04-01,NaT,NaN,0.0,10.080000,72.000000,NaN,NaN,0
1,1419.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,NaT,NaT,...,70.000000,2019-04-01,NaT,NaN,0.0,9.800000,70.000000,NaN,NaN,0
2,781.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,NaT,NaT,...,96.000000,2019-04-01,NaT,NaN,0.0,13.440000,96.000000,NaN,NaN,0
3,853.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,NaT,NaT,...,75.000000,2019-04-01,NaT,NaN,0.0,10.500000,75.000000,NaN,NaN,0
4,160.0,EVALUACION ACUMULATIVA,10,1,True,2019,percent,14.0,NaT,NaT,...,64.000000,2019-04-22,NaT,NaN,0.0,8.960000,64.000000,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68154,722.0,EVALUACION DE CIERRE,190,1,True,2022,points,4.0,2022-06-03,NaT,...,0.027130,2022-06-01,NaT,online_quiz,0.0,0.027130,0.678261,True,0.0,0
68155,722.0,PARCIAL 3,190,1,True,2022,percent,11.0,2022-05-02,NaT,...,0.074609,2022-05-04,NaT,online_upload,0.0,0.074609,0.678261,True,0.0,0
68156,722.0,PARCIAL 4,190,1,True,2022,percent,11.0,NaT,NaT,...,0.074609,2022-06-07,NaT,NaN,0.0,0.074609,0.678261,True,0.0,0
68157,722.0,SIMULACRO 3,190,1,True,2022,percent,4.0,NaT,2022-04-22,...,0.027130,2022-04-26,2022-04-22,online_upload,0.0,0.027130,0.678261,True,0.0,0
